# V2 School Environment Features

## 목적

02번 노트북에서 만든 **연도별 학교환경 테이블(EDSS 2021~2025, 보정 완료)**을 이용해
아파트 단지별·연도별 학교환경 Feature를 생성한다.

V1은 현재 학교 목록 기준의 최근접 학교 거리만 사용했다.
V2는 **거래 시점에 실제 운영 중이던 학교**를 기준으로 거리와 학교 규모·과밀도를 계산한다.

### 시점 원칙 (미래정보 누출 방지)

- 학교 통계는 매년 **4월 1일 기준**으로 집계되어 학교알리미에 **5월 공시**된다.
- 거래월이 **6월 이상이면 당해연도**, 1~5월이면 **전년도** 학교 데이터를 사용한다.
- EDSS는 2025년까지 제공되므로 2026년 거래는 2025년 데이터를 사용한다.

### B2C 웹앱 활용

- 최근접 초·중학교의 학교명·거리·학급당학생수는 단지 상세 화면의 교육환경 카드로 그대로 표시한다.
- 모델 입력용 수치 Feature와 화면 표시용 컬럼(학교명)을 구분해 저장한다.

### 산출물

- `school_features_by_location_2021_2025.parquet`: 단지 좌표 × 학교기준연도 Feature (웹앱 조회용)
- `df_fe_school_v2.parquet`: 거래 단위 Feature (모델 병합용, `transaction_id` Key)

In [1]:
# ============================================================
# 프로젝트 경로 설정
# - 현재 노트북 위치와 관계없이 data 폴더가 있는 프로젝트 루트를 찾는다.
# - 02번 노트북에서 저장한 학교환경 테이블과 V1 거래 데이터를 사용한다.
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.neighbors import BallTree


def find_project_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "data").exists():
            return path

    raise FileNotFoundError(
        "프로젝트 루트에서 data 폴더를 찾을 수 없습니다."
    )


PROJECT_DIR = find_project_root()

PROCESSED_DIR = (
    PROJECT_DIR
    / "data"
    / "v2_5year_seoul"
    / "processed"
)

SCHOOL_DIR = PROCESSED_DIR / "school"

EARTH_RADIUS_M = 6_371_000


print("PROJECT_DIR:", PROJECT_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)

PROJECT_DIR: C:\Users\uk\Apartment price
PROCESSED_DIR: C:\Users\uk\Apartment price\data\v2_5year_seoul\processed


In [2]:
# ============================================================
# 학교환경 테이블 및 거래 데이터 로드
# - 학교: 02번 노트북의 EDSS 2021~2025 최종 테이블 (학생수·위치 보정 완료)
# - 거래: V1 최종 데이터에서 Key·좌표·거래시점 컬럼만 사용한다.
# ============================================================

school_env = pd.read_parquet(
    SCHOOL_DIR
    / "edss_school_environment_2021_2025.parquet"
)


deals = (
    pd.read_parquet(
        PROCESSED_DIR / "df_final_v1.parquet",
        columns=[
            "transaction_id",
            "단지일련번호",
            "단지명",
            "시군구명",
            "위도",
            "경도",
            "거래연월",
        ],
    )
    .astype(
        {
            "거래연월": int,
            "단지명": str,
            "시군구명": str,
        }
    )
)


print("학교환경:", school_env.shape)
print("거래:", deals.shape)

print(
    "\n학교 기준연도:",
    sorted(school_env["기준연도"].unique()),
)

print(
    "거래 기간:",
    deals["거래연월"].min(),
    "~",
    deals["거래연월"].max(),
)

print(
    "좌표 결측:",
    deals[["위도", "경도"]].isna().sum().sum(),
)

학교환경: (6561, 17)
거래: (234302, 7)

학교 기준연도: [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
거래 기간: 202107 ~ 202606
좌표 결측: 0


In [3]:
# ============================================================
# 학교 학생수 전년 대비 증감률 계산
# - 학교 단위 수요 추세(학생 유입/유출)를 나타낸다.
# - 동일 개방ID의 직전 연도 값이 있을 때만 계산한다.
# - 2021년(EDSS 첫해)과 신설 학교의 첫해는 결측으로 남는다.
# ============================================================

school_env = (
    school_env
    .sort_values(["개방ID", "기준연도"])
    .reset_index(drop=True)
)


previous = (
    school_env
    .groupby("개방ID")[["기준연도", "학생수"]]
    .shift(1)
)


has_previous_year = (
    previous["기준연도"]
    == school_env["기준연도"] - 1
)


school_env["학생수증감률"] = np.where(
    has_previous_year,
    (
        school_env["학생수"]
        / previous["학생수"]
        - 1
    )
    * 100,
    np.nan,
)


display(
    school_env
    .groupby("기준연도")["학생수증감률"]
    .agg(["count", "mean", "median"])
    .round(2)
)

,count,mean,median
기준연도,,,
2021,0,NaN,NaN
2022,1311,-2.72,-2.97
2023,1313,-3.13,-3.14
2024,1307,-1.42,-3.38
2025,1309,-3.41,-3.68


## 추가 학군 지표 (EDSS)

EDSS 원본 7종 중 학군 지표로 사용할 수 있는 항목을 데이터 품질 기준으로 선별했다.

| 지표 | 원본 | 사용 | 사유 |
|---|---|---|---|
| 학생 순전입률 | 학생변동현황 | ✅ | 학군 이동 수요를 직접 반영 |
| 학생 1인당 교수학습공간·체육장 면적 | 학교개황 | ✅ | 2024년 전체 0 → 직전 연도 값으로 보완 |
| 교사동 노후도 | 건물현황 | ✅ | 건축년도 결측 없음 |
| 도서관 도서수·좌석 | 도서관운영현황 | ❌ | 초·중학교는 전부 0 (고등학교만 보고) |
| 기간제교사 비율 | 교원현황 | ❌ | 서울 전 학교 0 |
| 정규교실수 | 학교개황 | ❌ | 중앙값 1·최대 8로 실제 교실 수가 아님 |
| 졸업생수 | 학교개황 | ❌ | 2021~2023 54% 0 |
| 건물 안전등급 | 건물현황 | ❌ | D·E등급 10건으로 변별력 없음 |

**시점:** 학생변동현황의 조사년도 Y는 직전 학년도(Y-1년 3월 ~ Y년 2월) 전입·전출을 집계한다.
조사년도 Y의 다른 지표와 같은 시점에 공시되므로 동일한 학교기준연도 규칙을 적용한다.

In [4]:
# ============================================================
# 학생 순전입률 계산 (학생변동현황)
# - 순전입률 = (도내·도외 전입 - 도내·도외 전출) / 학년도 시작 학생수 × 100
# - 학년도 시작 학생수 = 3월 학생수(2학년 이상) + 입학생
# - 학군 수요가 높은 학교일수록 전입이 전출보다 많다.
# ============================================================

EDSS_DIR = PROCESSED_DIR / "edss"

MOVEMENT_COLS = [
    "조사년도3월학생수",
    "입학증가수",
    "도내전입증가수",
    "도외전입증가수",
    "도내전출감소수",
    "도외전출감소수",
]


student_movement = pd.read_parquet(
    EDSS_DIR / "student_movement_2021_2025.parquet",
    columns=["조사년도", "개방ID"] + MOVEMENT_COLS,
)

student_movement[MOVEMENT_COLS] = (
    student_movement[MOVEMENT_COLS]
    .apply(pd.to_numeric, errors="coerce")
)


movement = (
    student_movement
    .groupby(["조사년도", "개방ID"])[MOVEMENT_COLS]
    .sum()
    .reset_index()
    .rename(columns={"조사년도": "기준연도"})
)

movement["학년도시작학생수"] = (
    movement["조사년도3월학생수"]
    + movement["입학증가수"]
)

movement["순전입학생수"] = (
    movement["도내전입증가수"]
    + movement["도외전입증가수"]
    - movement["도내전출감소수"]
    - movement["도외전출감소수"]
)

movement["학생순전입률"] = np.where(
    movement["학년도시작학생수"] > 0,
    movement["순전입학생수"]
    / movement["학년도시작학생수"]
    * 100,
    np.nan,
)


display(
    movement
    .groupby("기준연도")["학생순전입률"]
    .describe()
    .round(2)
)

,count,mean,std,min,25%,50%,75%,max
기준연도,,,,,,,,
2021,11895.0,6.05,566.24,-100.0,-2.36,-0.33,0.87,61700.00
2022,11937.0,0.59,20.75,-100.0,-2.27,-0.33,0.91,952.11
2023,11954.0,-0.03,14.13,-100.0,-2.23,-0.42,0.69,754.95
2024,11964.0,0.23,23.42,-100.0,-2.60,-0.53,0.63,1460.00
2025,11960.0,0.20,23.12,-100.0,-2.59,-0.51,0.69,1085.07


In [5]:
# ============================================================
# 학생 1인당 시설 면적 계산 (학교개황)
# - 교수학습공간·체육장 면적은 0을 결측으로 처리한다.
# - 2024년은 전 학교 면적이 0으로 제공되어 같은 학교의 직전 연도 값으로 보완한다.
#   (과거 값만 사용하므로 미래정보 누출이 없다.)
# - 학생수는 02번에서 방송통신 부설 학교를 보정한 값을 사용한다.
# ============================================================

AREA_COLS = {
    "유초중등학교개황_교수학습공간면적": "교수학습공간면적",
    "유초중등학교개황_학교용지체육장면적": "체육장면적",
}


facility = (
    pd.read_parquet(
        EDSS_DIR / "school_overview_2021_2025.parquet",
        columns=["조사년도", "개방ID"] + list(AREA_COLS),
    )
    .rename(columns={"조사년도": "기준연도", **AREA_COLS})
)

area_cols = list(AREA_COLS.values())

facility[area_cols] = (
    facility[area_cols]
    .apply(pd.to_numeric, errors="coerce")
    .replace(0, np.nan)
)


print("보완 전 연도별 결측률:")
display(
    facility
    .groupby("기준연도")[area_cols]
    .apply(lambda x: x.isna().mean())
    .round(2)
)


facility = facility.sort_values(["개방ID", "기준연도"])

facility[area_cols] = (
    facility
    .groupby("개방ID")[area_cols]
    .ffill()
)


print("보완 후 연도별 결측률:")
display(
    facility
    .groupby("기준연도")[area_cols]
    .apply(lambda x: x.isna().mean())
    .round(2)
)

보완 전 연도별 결측률:


,교수학습공간면적,체육장면적
기준연도,,
2021,0.24,0.3
2022,0.24,0.3
2023,0.24,0.3
2024,1.00,1.0
2025,0.24,0.3


보완 후 연도별 결측률:


,교수학습공간면적,체육장면적
기준연도,,
2021,0.24,0.30
2022,0.23,0.29
2023,0.23,0.29
2024,0.24,0.29
2025,0.23,0.29


In [6]:
# ============================================================
# 교사동 노후도 계산 (건물현황)
# - 건물용도가 '교사'인 건물만 사용한다. (창고·경비실 등 제외)
# - 연면적 가중 평균 경과년수 = Σ(연면적 × (기준연도 - 건축년도)) / Σ연면적
# ============================================================

buildings = (
    pd.read_parquet(
        EDSS_DIR / "building_status_2021_2025.parquet",
        columns=["조사년도", "개방ID", "건물용도명", "건축년도", "건물연면적"],
    )
    .rename(columns={"조사년도": "기준연도"})
)

buildings["건축년도"] = pd.to_numeric(buildings["건축년도"], errors="coerce")
buildings["건물연면적"] = pd.to_numeric(buildings["건물연면적"], errors="coerce")


classroom_buildings = buildings[
    (buildings["건물용도명"] == "교사")
    & buildings["건축년도"].between(1880, buildings["기준연도"])
    & (buildings["건물연면적"] > 0)
].copy()

classroom_buildings["가중경과년수"] = (
    (classroom_buildings["기준연도"] - classroom_buildings["건축년도"])
    * classroom_buildings["건물연면적"]
)


building_age = (
    classroom_buildings
    .groupby(["기준연도", "개방ID"])[["가중경과년수", "건물연면적"]]
    .sum()
    .reset_index()
)

building_age["교사동_평균경과년수"] = (
    building_age["가중경과년수"]
    / building_age["건물연면적"]
)


display(
    building_age["교사동_평균경과년수"]
    .describe()
    .round(1)
)

count    78910.0
mean        29.2
std         14.5
min          0.0
25%         18.0
50%         29.0
75%         40.0
max        102.0
Name: 교사동_평균경과년수, dtype: float64

In [7]:
# ============================================================
# 학교 단위 추가 학군 지표 병합
# - 학교환경 테이블(기준연도 + 개방ID)에 순전입·시설·노후도 지표를 연결한다.
# - 학생 1인당 면적은 보정된 학생수 기준으로 계산한다.
# ============================================================

school_env = (
    school_env
    .merge(
        movement[["기준연도", "개방ID", "학년도시작학생수", "순전입학생수", "학생순전입률"]],
        on=["기준연도", "개방ID"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        facility[["기준연도", "개방ID"] + area_cols],
        on=["기준연도", "개방ID"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        building_age[["기준연도", "개방ID", "교사동_평균경과년수"]],
        on=["기준연도", "개방ID"],
        how="left",
        validate="one_to_one",
    )
)


school_env["학생1인당_교수학습공간"] = (
    school_env["교수학습공간면적"]
    / school_env["학생수"]
)

school_env["학생1인당_체육장면적"] = (
    school_env["체육장면적"]
    / school_env["학생수"]
)


NEW_SCHOOL_COLS = [
    "학생순전입률",
    "학생1인당_교수학습공간",
    "학생1인당_체육장면적",
    "교사동_평균경과년수",
]


print("학교 단위 추가 지표 결측률:")
display(
    school_env
    .groupby("학교급구분")[NEW_SCHOOL_COLS]
    .apply(lambda x: x.isna().mean())
    .round(3)
)

display(
    school_env[NEW_SCHOOL_COLS]
    .describe()
    .T
    .round(2)
)

학교 단위 추가 지표 결측률:


,학생순전입률,학생1인당_교수학습공간,학생1인당_체육장면적,교사동_평균경과년수
학교급구분,,,,
고등학교,0.001,0.001,0.056,0.007
중학교,0.002,0.010,0.225,0.015
초등학교,0.003,0.006,0.043,0.006


,count,mean,std,min,25%,50%,75%,max
학생순전입률,6548.0,0.02,23.04,-54.97,-2.47,-0.82,0.55,1460.00
학생1인당_교수학습공간,6522.0,10.13,13.92,2.03,6.41,8.76,11.85,1001.44
학생1인당_체육장면적,5905.0,13.54,13.66,0.14,5.39,9.79,17.57,350.44
교사동_평균경과년수,6502.0,33.74,13.28,0.00,24.66,35.00,42.00,78.78


In [8]:
# ============================================================
# 거래별 학교기준연도 결정
# - 학교알리미 5월 공시 이후(6월~)는 당해연도 데이터를 사용한다.
# - 1~5월 거래는 아직 공시 전이므로 전년도 데이터를 사용한다.
# - EDSS 범위(2021~2025)를 벗어나는 연도는 가장 가까운 과거 연도로 제한한다.
# ============================================================

SCHOOL_RELEASE_MONTH = 6

SCHOOL_YEAR_MIN = int(school_env["기준연도"].min())
SCHOOL_YEAR_MAX = int(school_env["기준연도"].max())


deals["거래년도"] = deals["거래연월"] // 100
deals["거래월"] = deals["거래연월"] % 100


deals["학교기준연도"] = (
    np.where(
        deals["거래월"] >= SCHOOL_RELEASE_MONTH,
        deals["거래년도"],
        deals["거래년도"] - 1,
    )
    .clip(SCHOOL_YEAR_MIN, SCHOOL_YEAR_MAX)
)


# 미래정보 검증: 학교 데이터는 반드시 거래 시점 이전에 공시된 연도여야 한다.
published_before_deal = (
    (deals["학교기준연도"] < deals["거래년도"])
    | (
        (deals["학교기준연도"] == deals["거래년도"])
        & (deals["거래월"] >= SCHOOL_RELEASE_MONTH)
    )
)

assert published_before_deal.all()


display(
    deals
    .groupby("학교기준연도")["거래연월"]
    .agg(["min", "max", "size"])
    .rename(columns={"size": "거래수"})
)

,min,max,거래수
학교기준연도,,,
2021,202107,202205,22967
2022,202206,202305,18613
2023,202306,202405,39650
2024,202406,202505,68445
2025,202506,202606,84627


In [9]:
# ============================================================
# Feature 계산 단위: 단지 좌표 × 학교기준연도
# - 같은 좌표의 거래는 같은 학교환경을 가지므로 고유 좌표 단위로 한 번만 계산한다.
# - 이후 거래 데이터에 좌표 + 학교기준연도로 다시 연결한다.
# ============================================================

locations = (
    deals[["위도", "경도"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

location_radians = np.radians(
    locations[["위도", "경도"]].to_numpy()
)


print("고유 좌표 수:", len(locations))
print(
    "단지 수:",
    deals["단지일련번호"].nunique(),
)

고유 좌표 수: 7820
단지 수: 7871


In [10]:
# ============================================================
# 연도별 학교환경 Feature 생성 함수
# - 최근접 초·중학교: 학교명, 거리, 학생수, 학급당학생수, 교원1인당학생수, 학생수증감률
# - 반경 내 학교 수: 초등학교 500m, 중학교·고등학교 1km
# - 반경 1km 초등학생 수 합계: 주변 학령인구(가족 수요) 대리 지표
# - 반경 1km 초등 순전입률: 주변 학교 전체의 학군 이동 수요
# - 해당 연도에 운영 중인 학교만 사용하므로 폐교·신설이 자동 반영된다.
# ============================================================

NEAREST_LEVELS = {
    "초등학교": "최근접초",
    "중학교": "최근접중",
}

NEAREST_VALUE_COLS = [
    "학생수",
    "학급당학생수",
    "교원1인당학생수",
    "학생수증감률",
    "학생순전입률",
    "학생1인당_교수학습공간",
    "학생1인당_체육장면적",
    "교사동_평균경과년수",
]

RADIUS_COUNT_RULES = [
    ("초등학교", 500, "반경500m_초등학교수"),
    ("중학교", 1000, "반경1km_중학교수"),
    ("고등학교", 1000, "반경1km_고등학교수"),
]


def build_school_tree(schools):
    return BallTree(
        np.radians(
            schools[["위도", "경도"]].to_numpy()
        ),
        metric="haversine",
    )


def build_school_features(year):

    schools_year = school_env[
        school_env["기준연도"] == year
    ]

    features = locations.copy()
    features["학교기준연도"] = year


    for level, prefix in NEAREST_LEVELS.items():

        schools = (
            schools_year[
                schools_year["학교급구분"] == level
            ]
            .reset_index(drop=True)
        )

        distances, indices = (
            build_school_tree(schools)
            .query(location_radians, k=1)
        )

        nearest = schools.iloc[indices[:, 0]]

        features[f"{prefix}_학교명"] = nearest["학교명"].to_numpy()
        features[f"{prefix}_거리_m"] = distances[:, 0] * EARTH_RADIUS_M

        for col in NEAREST_VALUE_COLS:
            features[f"{prefix}_{col}"] = nearest[col].to_numpy()


    for level, radius_m, feature_name in RADIUS_COUNT_RULES:

        schools = schools_year[
            schools_year["학교급구분"] == level
        ]

        features[feature_name] = (
            build_school_tree(schools)
            .query_radius(
                location_radians,
                r=radius_m / EARTH_RADIUS_M,
                count_only=True,
            )
        )


    elementary = (
        schools_year[
            schools_year["학교급구분"] == "초등학교"
        ]
        .reset_index(drop=True)
    )

    elementary_students = elementary["학생수"].to_numpy()

    neighbor_indices = (
        build_school_tree(elementary)
        .query_radius(
            location_radians,
            r=1000 / EARTH_RADIUS_M,
        )
    )

    features["반경1km_초등학생수"] = [
        elementary_students[idx].sum()
        for idx in neighbor_indices
    ]


    # 순전입 지표가 있는 학교만 분자·분모에 포함한다.
    has_movement = elementary["학년도시작학생수"].notna().to_numpy()
    net_movers = elementary["순전입학생수"].fillna(0).to_numpy()
    base_students = elementary["학년도시작학생수"].fillna(0).to_numpy()

    radius_net = []

    for idx in neighbor_indices:
        idx = idx[has_movement[idx]]
        base = base_students[idx].sum()
        radius_net.append(
            net_movers[idx].sum() / base * 100
            if base > 0
            else np.nan
        )

    features["반경1km_초등순전입률"] = radius_net

    return features

In [11]:
# ============================================================
# 단지 좌표 × 학교기준연도 Feature 생성
# - 거래에 실제로 필요한 학교기준연도만 계산한다.
# - 결과는 웹앱에서 단지별 최신 교육환경을 조회하는 테이블로도 사용한다.
# ============================================================

SCHOOL_YEARS = sorted(
    deals["학교기준연도"].unique()
)


school_location_features = pd.concat(
    [
        build_school_features(year)
        for year in SCHOOL_YEARS
    ],
    ignore_index=True,
)


print("학교기준연도:", SCHOOL_YEARS)
print("좌표 × 연도 Feature:", school_location_features.shape)

display(
    school_location_features.head()
)

학교기준연도: [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
좌표 × 연도 Feature: (39100, 28)


,위도,경도,학교기준연도,최근접초_학교명,최근접초_거리_m,최근접초_학생수,최근접초_학급당학생수,최근접초_교원1인당학생수,최근접초_학생수증감률,최근접초_학생순전입률,...,최근접중_학생수증감률,최근접중_학생순전입률,최근접중_학생1인당_교수학습공간,최근접중_학생1인당_체육장면적,최근접중_교사동_평균경과년수,반경500m_초등학교수,반경1km_중학교수,반경1km_고등학교수,반경1km_초등학생수,반경1km_초등순전입률
0,37.575631,126.961060,2021,서울독립문초등학교,184.266445,487,18.730769,10.145833,NaN,-1.431493,...,NaN,-0.536193,3.978320,NaN,61.000000,1,3,3,1889,-3.401701
1,37.574417,126.968881,2021,서울매동초등학교,382.997401,192,13.714286,7.111111,NaN,-3.587444,...,NaN,-0.233100,4.609170,8.384279,59.000000,1,4,4,1681,-3.519553
2,37.570696,126.963609,2021,서울금화초등학교,215.940049,519,19.961538,13.307692,NaN,-5.043478,...,NaN,0.353357,9.104869,7.490637,64.629280,1,4,4,2565,-2.770689
3,37.572614,126.972273,2021,서울덕수초등학교,444.822543,483,21.000000,13.416667,NaN,-3.777336,...,NaN,0.980392,11.963470,17.378995,49.484030,1,4,4,1194,-4.304381
4,37.580323,127.011785,2021,서울명신초등학교,280.621824,336,16.800000,11.586207,NaN,-2.400000,...,NaN,-1.342282,17.209412,NaN,34.395981,1,3,4,2448,-2.175602


In [12]:
# ============================================================
# 단지 좌표 × 연도 Feature 검증
# - Key(좌표 + 학교기준연도) 중복이 없어야 한다.
# - 결측 허용 컬럼(증감률·추가 학군 지표)을 제외한 모든 Feature는 결측이 없어야 한다.
# - 학생수증감률 결측은 2021년(첫해)과 신설 학교 첫해에만 허용한다.
# - 추가 학군 지표는 원천 결측이 있어 결측률 10% 미만만 허용한다.
# ============================================================

LOCATION_KEY = [
    "위도",
    "경도",
    "학교기준연도",
]

GROWTH_COLS = [
    "최근접초_학생수증감률",
    "최근접중_학생수증감률",
]

EXTRA_NULLABLE_COLS = [
    f"{prefix}_{col}"
    for prefix in NEAREST_LEVELS.values()
    for col in NEW_SCHOOL_COLS
] + ["반경1km_초등순전입률"]

NULLABLE_COLS = GROWTH_COLS + EXTRA_NULLABLE_COLS


key_duplicates = (
    school_location_features
    .duplicated(subset=LOCATION_KEY)
    .sum()
)

missing = (
    school_location_features
    .drop(columns=NULLABLE_COLS)
    .isna()
    .sum()
    .sum()
)


print("Key 중복:", key_duplicates)
print("결측 허용 컬럼 외 결측:", missing)

print("\n학생수증감률 결측 (연도별):")
display(
    school_location_features
    .groupby("학교기준연도")[GROWTH_COLS]
    .apply(lambda x: x.isna().sum())
)


assert key_duplicates == 0
assert missing == 0

# 2022년 이후 증감률 결측은 신설 학교 첫해에 해당하는 소수 좌표뿐이어야 한다.
assert (
    school_location_features.loc[
        school_location_features["학교기준연도"] > SCHOOL_YEAR_MIN,
        GROWTH_COLS,
    ]
    .isna()
    .mean()
    .max()
    < 0.05
)

print("\n추가 학군 지표 결측률:")
extra_missing = (
    school_location_features[EXTRA_NULLABLE_COLS]
    .isna()
    .mean()
)
display(extra_missing.round(3).to_frame("결측률"))

# 중학교는 약 22%가 체육장 면적을 0으로 보고해(원인 미확인) 결측 처리했으므로 별도 기준을 둔다.
HIGH_MISSING_ALLOWED = {
    "최근접중_학생1인당_체육장면적": 0.25,
}

missing_limit = pd.Series(0.10, index=extra_missing.index)
missing_limit.update(pd.Series(HIGH_MISSING_ALLOWED))

assert (extra_missing < missing_limit).all()


display(
    school_location_features
    .describe()
    .T
    .round(1)
)

Key 중복: 0
결측 허용 컬럼 외 결측: 0

학생수증감률 결측 (연도별):


,최근접초_학생수증감률,최근접중_학생수증감률
학교기준연도,,
2021,7820,7820
2022,6,9
2023,4,0
2024,3,10
2025,24,0



추가 학군 지표 결측률:


,결측률
최근접초_학생순전입률,0.001
최근접초_학생1인당_교수학습공간,0.003
최근접초_학생1인당_체육장면적,0.033
최근접초_교사동_평균경과년수,0.003
최근접중_학생순전입률,0.001
최근접중_학생1인당_교수학습공간,0.003
최근접중_학생1인당_체육장면적,0.189
최근접중_교사동_평균경과년수,0.009
반경1km_초등순전입률,0.002


,count,mean,std,min,25%,50%,75%,max
위도,39100.0,37.5,0.0,37.4,37.5,37.5,37.6,37.7
경도,39100.0,127.0,0.1,126.8,126.9,127.0,127.1,127.2
학교기준연도,39100.0,2023.0,1.4,2021.0,2022.0,2023.0,2024.0,2025.0
최근접초_거리_m,39100.0,368.3,176.2,23.3,236.7,350.6,473.0,1768.8
최근접초_학생수,39100.0,678.6,360.6,66.0,426.0,603.0,864.0,2129.0
최근접초_학급당학생수,39100.0,20.4,3.8,8.1,18.3,20.1,22.0,34.6
최근접초_교원1인당학생수,39100.0,13.1,3.1,4.1,11.2,12.8,14.5,26.4
최근접초_학생수증감률,31243.0,-4.7,6.9,-71.9,-8.3,-5.1,-1.8,138.7
최근접초_학생순전입률,39060.0,-1.1,10.3,-37.6,-3.9,-2.1,0.2,547.6
최근접초_학생1인당_교수학습공간,39001.0,9.0,4.5,2.4,5.8,7.9,10.8,36.7


In [13]:
# ============================================================
# V1 최근접 학교 거리와 비교 검증
# - V1은 현재(최신) 학교 위치 목록으로 계산했다.
# - V2의 최신 연도(2025) 거리는 V1과 거의 같아야 한다.
# - 과거 연도에서 차이가 나는 좌표는 폐교·신설 학교가 반영된 결과다.
# ============================================================

v1_school = pd.read_parquet(
    PROCESSED_DIR / "df_fe_school.parquet",
    columns=[
        "transaction_id",
        "초등학교거리_m",
        "중학교거리_m",
    ],
)


v1_location = (
    deals[["transaction_id", "위도", "경도"]]
    .merge(
        v1_school,
        on="transaction_id",
        validate="one_to_one",
    )
    .drop_duplicates(subset=["위도", "경도"])
)


compare = (
    school_location_features[
        ["위도", "경도", "학교기준연도", "최근접초_거리_m", "최근접중_거리_m"]
    ]
    .merge(
        v1_location,
        on=["위도", "경도"],
        validate="many_to_one",
    )
)


compare["초_차이_m"] = (
    compare["최근접초_거리_m"]
    - compare["초등학교거리_m"]
).abs()

compare["중_차이_m"] = (
    compare["최근접중_거리_m"]
    - compare["중학교거리_m"]
).abs()


display(
    compare
    .groupby("학교기준연도")
    .agg(
        초_10m이상차이비율=("초_차이_m", lambda x: (x > 10).mean() * 100),
        중_10m이상차이비율=("중_차이_m", lambda x: (x > 10).mean() * 100),
        초_차이중앙값=("초_차이_m", "median"),
    )
    .round(2)
)

,초_10m이상차이비율,중_10m이상차이비율,초_차이중앙값
학교기준연도,,,
2021,1.05,0.52,0.0
2022,0.97,0.42,0.0
2023,0.75,0.42,0.0
2024,0.60,0.31,0.0
2025,0.00,0.31,0.0


In [14]:
# ============================================================
# 폐교 반영 사례 확인: 서울화양초등학교 (2023년 폐교)
# - 2022년까지 화양초가 최근접이던 단지가
#   이후 연도에는 다른 학교로 바뀌는지 확인한다.
# - V1(현재 학교 목록)에서는 화양초가 처음부터 없어 과거 거리가 과대 계산됐다.
# ============================================================

hwayang_locations = school_location_features.loc[
    (school_location_features["최근접초_학교명"] == "서울화양초등학교"),
    ["위도", "경도"],
].drop_duplicates()


hwayang_case = (
    school_location_features
    .merge(
        hwayang_locations,
        on=["위도", "경도"],
    )
    .merge(
        deals[["위도", "경도", "단지명"]].drop_duplicates(subset=["위도", "경도"]),
        on=["위도", "경도"],
    )
)


print("화양초가 최근접이었던 좌표 수:", len(hwayang_locations))

display(
    hwayang_case[
        hwayang_case["단지명"] == hwayang_case["단지명"].iloc[0]
    ][
        [
            "단지명",
            "학교기준연도",
            "최근접초_학교명",
            "최근접초_거리_m",
            "최근접초_학급당학생수",
        ]
    ]
    .round(1)
)

화양초가 최근접이었던 좌표 수: 10


,단지명,학교기준연도,최근접초_학교명,최근접초_거리_m,최근접초_학급당학생수
0,광진코지웰,2021,서울화양초등학교,304.4,14.6
10,광진코지웰,2022,서울화양초등학교,304.4,10.5
20,광진코지웰,2023,서울장안초등학교,316.2,18.9
30,광진코지웰,2024,서울장안초등학교,316.2,17.9
40,광진코지웰,2025,서울장안초등학교,316.2,17.1


In [15]:
# ============================================================
# 거래 단위 학교 Feature 생성
# - 거래 좌표 + 학교기준연도로 단지 Feature를 연결한다.
# - transaction_id를 Key로 V1 최종 데이터와 병합할 수 있도록 만든다.
# ============================================================

df_fe_school_v2 = (
    deals[
        [
            "transaction_id",
            "위도",
            "경도",
            "학교기준연도",
        ]
    ]
    .merge(
        school_location_features,
        on=LOCATION_KEY,
        how="left",
        validate="many_to_one",
    )
    .drop(columns=["위도", "경도"])
)


print("거래 단위 Feature:", df_fe_school_v2.shape)

print(
    "transaction_id 중복:",
    df_fe_school_v2["transaction_id"].duplicated().sum(),
)

print(
    "결측 허용 컬럼 외 결측:",
    df_fe_school_v2
    .drop(columns=NULLABLE_COLS)
    .isna()
    .sum()
    .sum(),
)


assert len(df_fe_school_v2) == len(deals)
assert df_fe_school_v2["transaction_id"].is_unique
assert (
    df_fe_school_v2
    .drop(columns=NULLABLE_COLS)
    .isna()
    .sum()
    .sum()
    == 0
)

거래 단위 Feature: (234302, 27)
transaction_id 중복: 0
결측 허용 컬럼 외 결측: 0


In [16]:
# ============================================================
# V2 학교 Feature 목록 정의
# - 모델 입력용 수치 Feature와 웹앱 표시용 컬럼을 분리해 관리한다.
# - 학교명은 화면 표시용이며 모델에는 사용하지 않는다.
# ============================================================

SCHOOL_V2_FEATURES = [

    # 최근접 초등학교
    "최근접초_거리_m",
    "최근접초_학생수",
    "최근접초_학급당학생수",
    "최근접초_교원1인당학생수",
    "최근접초_학생수증감률",
    "최근접초_학생순전입률",
    "최근접초_학생1인당_교수학습공간",
    "최근접초_학생1인당_체육장면적",
    "최근접초_교사동_평균경과년수",

    # 최근접 중학교
    "최근접중_거리_m",
    "최근접중_학생수",
    "최근접중_학급당학생수",
    "최근접중_교원1인당학생수",
    "최근접중_학생수증감률",
    "최근접중_학생순전입률",
    "최근접중_학생1인당_교수학습공간",
    "최근접중_학생1인당_체육장면적",
    "최근접중_교사동_평균경과년수",

    # 학교 공급 / 학령 수요
    "반경500m_초등학교수",
    "반경1km_중학교수",
    "반경1km_고등학교수",
    "반경1km_초등학생수",
    "반경1km_초등순전입률",
]

SCHOOL_V2_DISPLAY_COLS = [
    "최근접초_학교명",
    "최근접중_학교명",
]


assert set(SCHOOL_V2_FEATURES + SCHOOL_V2_DISPLAY_COLS) == (
    set(df_fe_school_v2.columns)
    - {"transaction_id", "학교기준연도"}
)


print("V2 학교 Feature 수:", len(SCHOOL_V2_FEATURES))

display(
    df_fe_school_v2[SCHOOL_V2_FEATURES]
    .corr()
    .round(2)
)

V2 학교 Feature 수: 23


,최근접초_거리_m,최근접초_학생수,최근접초_학급당학생수,최근접초_교원1인당학생수,최근접초_학생수증감률,최근접초_학생순전입률,최근접초_학생1인당_교수학습공간,최근접초_학생1인당_체육장면적,최근접초_교사동_평균경과년수,최근접중_거리_m,...,최근접중_학생수증감률,최근접중_학생순전입률,최근접중_학생1인당_교수학습공간,최근접중_학생1인당_체육장면적,최근접중_교사동_평균경과년수,반경500m_초등학교수,반경1km_중학교수,반경1km_고등학교수,반경1km_초등학생수,반경1km_초등순전입률
최근접초_거리_m,1.00,-0.06,-0.08,-0.09,-0.02,-0.01,0.06,0.05,0.12,0.16,...,-0.01,0.02,0.03,0.03,-0.04,-0.58,-0.15,-0.16,-0.19,-0.02
최근접초_학생수,-0.06,1.00,0.79,0.68,0.26,0.13,-0.71,-0.55,-0.17,-0.13,...,0.11,0.05,-0.23,-0.26,-0.06,-0.02,0.08,0.05,0.42,0.23
최근접초_학급당학생수,-0.08,0.79,1.00,0.93,0.36,0.14,-0.75,-0.57,-0.09,-0.16,...,0.08,0.05,-0.23,-0.24,0.02,0.05,0.10,0.10,0.41,0.26
최근접초_교원1인당학생수,-0.09,0.68,0.93,1.00,0.37,0.15,-0.71,-0.54,-0.08,-0.17,...,0.09,0.05,-0.20,-0.22,0.02,0.08,0.11,0.12,0.38,0.23
최근접초_학생수증감률,-0.02,0.26,0.36,0.37,1.00,0.38,-0.31,-0.18,-0.09,-0.05,...,0.04,0.11,-0.05,-0.05,-0.00,0.03,0.03,0.01,0.15,0.25
최근접초_학생순전입률,-0.01,0.13,0.14,0.15,0.38,1.00,-0.11,-0.07,-0.07,-0.04,...,0.11,0.31,-0.05,-0.05,-0.05,0.01,0.04,0.06,0.12,0.50
최근접초_학생1인당_교수학습공간,0.06,-0.71,-0.75,-0.71,-0.31,-0.11,1.00,0.66,0.06,0.13,...,-0.05,0.00,0.18,0.18,-0.01,0.01,-0.05,-0.01,-0.32,-0.18
최근접초_학생1인당_체육장면적,0.05,-0.55,-0.57,-0.54,-0.18,-0.07,0.66,1.00,0.27,0.05,...,-0.06,-0.02,0.14,0.15,0.03,0.02,-0.03,0.01,-0.21,-0.07
최근접초_교사동_평균경과년수,0.12,-0.17,-0.09,-0.08,-0.09,-0.07,0.06,0.27,1.00,-0.06,...,-0.02,-0.00,0.07,0.09,0.17,-0.05,0.02,-0.06,-0.10,-0.03
최근접중_거리_m,0.16,-0.13,-0.16,-0.17,-0.05,-0.04,0.13,0.05,-0.06,1.00,...,-0.01,-0.03,-0.03,0.01,0.02,-0.20,-0.42,-0.20,-0.21,-0.11


In [17]:
# ============================================================
# B2C 웹앱 교육환경 카드 예시
# - 거래가 가장 많은 단지의 최신 학교기준연도 값을 화면 문구 형태로 출력한다.
# - 도보 시간은 분속 67m(시속 4km) 기준으로 환산한다.
# ============================================================

WALK_M_PER_MIN = 67


sample = (
    deals
    .groupby(["단지일련번호", "단지명", "시군구명"])
    .agg(
        거래수=("transaction_id", "size"),
        위도=("위도", "first"),
        경도=("경도", "first"),
    )
    .sort_values("거래수", ascending=False)
    .reset_index()
    .iloc[0]
)


card = school_location_features[
    (school_location_features["위도"] == sample["위도"])
    & (school_location_features["경도"] == sample["경도"])
    & (school_location_features["학교기준연도"] == SCHOOL_YEAR_MAX)
].iloc[0]


print(f"🏢 {sample['단지명']} ({sample['시군구명']}) · {SCHOOL_YEAR_MAX}년 학교 기준\n")

for prefix, label in [("최근접초", "초등학교"), ("최근접중", "중학교")]:
    print(
        f"📚 {label}: {card[f'{prefix}_학교명']} "
        f"· {card[f'{prefix}_거리_m']:.0f}m (도보 약 {card[f'{prefix}_거리_m'] / WALK_M_PER_MIN:.0f}분) "
        f"· 학급당 {card[f'{prefix}_학급당학생수']:.1f}명 "
        f"· 학생수 {card[f'{prefix}_학생수']:,}명 ({card[f'{prefix}_학생수증감률']:+.1f}%)"
    )
    print(
        f"   └ 순전입률 {card[f'{prefix}_학생순전입률']:+.1f}% "
        f"· 1인당 교수학습공간 {card[f'{prefix}_학생1인당_교수학습공간']:.1f}㎡ "
        f"· 교사동 {card[f'{prefix}_교사동_평균경과년수']:.0f}년차"
    )

print(
    f"\n🏫 주변 학교: 초등학교 {card['반경500m_초등학교수']}개(500m) · "
    f"중학교 {card['반경1km_중학교수']}개 · 고등학교 {card['반경1km_고등학교수']}개(1km)"
)

print(
    f"🚚 주변 초등학교 순전입률(1km): {card['반경1km_초등순전입률']:+.1f}%"
)

🏢 헬리오시티 (송파구) · 2025년 학교 기준

📚 초등학교: 서울가락초등학교 · 293m (도보 약 4분) · 학급당 21.8명 · 학생수 1,133명 (-7.8%)
   └ 순전입률 +1.1% · 1인당 교수학습공간 6.2㎡ · 교사동 28년차
📚 중학교: 해누리중학교 · 443m (도보 약 7분) · 학급당 28.4명 · 학생수 853명 (+20.1%)
   └ 순전입률 -2.0% · 1인당 교수학습공간 8.8㎡ · 교사동 6년차

🏫 주변 학교: 초등학교 2개(500m) · 중학교 4개 · 고등학교 5개(1km)
🚚 주변 초등학교 순전입률(1km): +1.2%


In [18]:
# ============================================================
# V2 학교 Feature 저장
# - 좌표 × 연도 테이블: 웹앱 단지 상세 조회 및 이후 신규 거래 Feature 생성용
# - 거래 단위 테이블: V2 Feature 병합 노트북에서 transaction_id로 결합
# ============================================================

LOCATION_FEATURE_PATH = (
    SCHOOL_DIR
    / "school_features_by_location_2021_2025.parquet"
)

DEAL_FEATURE_PATH = (
    PROCESSED_DIR
    / "df_fe_school_v2.parquet"
)


school_location_features.to_parquet(
    LOCATION_FEATURE_PATH,
    index=False,
)

df_fe_school_v2.to_parquet(
    DEAL_FEATURE_PATH,
    index=False,
)


for path in [LOCATION_FEATURE_PATH, DEAL_FEATURE_PATH]:

    check = pd.read_parquet(path)

    print(
        f"✅ {path.name}: {check.shape}"
    )

✅ school_features_by_location_2021_2025.parquet: (39100, 28)
✅ df_fe_school_v2.parquet: (234302, 27)
